In [1]:
import pandas as pd
import torch
import lightning as pl
import wandb
from torchmetrics.regression import  MeanAbsoluteError,MeanSquaredLogError,MeanAbsolutePercentageError,MeanSquaredError
from lightning.pytorch.loggers import WandbLogger

from configs import configs

## Завантаження даних

In [2]:
x_train = pd.read_csv("data/processed_data/x_train.csv").astype(float)
y_train = pd.read_csv("data/processed_data/y_train.csv").astype(float)

x_val = pd.read_csv("data/processed_data/x_val.csv").astype(float)
y_val = pd.read_csv("data/processed_data/y_val.csv").astype(float)

In [3]:
x_train.shape


(6531, 7)

In [4]:
class BicicleDataset(torch.utils.data.Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return torch.tensor(self.x.iloc[idx].values, dtype=torch.float32), torch.tensor(self.y.iloc[idx].values, dtype=torch.float32)

In [5]:
train_dataset = BicicleDataset(x_train, y_train)
val_dataset = BicicleDataset(x_val, y_val)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=configs["batch_size"], shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=configs["batch_size"], shuffle=False)

## Створення моделі

In [6]:
class CustomModel(pl.LightningModule):
    def __init__(self, lr):
        super().__init__()
        self.model = torch.nn.Sequential(
            torch.nn.Linear(7, 16),
            torch.nn.ReLU(),
            torch.nn.Linear(16, 1),
        )
        
        self.train_mae = MeanAbsoluteError()
        self.val_mae = MeanAbsoluteError()
        self.train_msle = MeanSquaredLogError()
        self.val_msle = MeanSquaredLogError()
        self.train_mape = MeanAbsolutePercentageError()
        self.val_mape = MeanAbsolutePercentageError()
        self.train_msqe = MeanSquaredError(squared=False)
        self.val_msqe = MeanSquaredError(squared=False)
         
        
        self.loss_fn = torch.nn.MSELoss()
        self.lr = lr

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)
        
        self.train_msqe.update(y_hat, y)
        self.train_mae.update(y_hat, y)
        self.train_msle.update(y_hat, y)
        self.train_mape.update(y_hat, y)
        
        self.log("train/loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)

        self.val_msqe.update(y_hat, y)
        self.val_mae.update(y_hat, y)
        self.val_msle.update(y_hat, y)
        self.val_mape.update(y_hat, y)

        self.log("val/loss", loss)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)
    
    def on_train_epoch_end(self):    
        self.log("train_msqe_epoch", self.train_msqe.compute())
        self.log("train_mae_epoch", self.train_mae.compute())
        self.log("train_msle_epoch", self.train_msle.compute())
        self.log("train_mape_epoch", self.train_mape.compute())
        
        
    
    def on_validation_epoch_end(self):
        self.log("val_msqe_epoch", self.val_msqe.compute())
        self.log("val_mae_epoch", self.val_mae.compute())
        self.log("val_msle_epoch", self.val_msle.compute())
        self.log("val_mape_epoch", self.val_mape.compute())                 


In [7]:
model = CustomModel(lr=configs["lr"])

## Створення логувальника WandB

In [8]:
configs["wandb_key"]

'80b641b6e2bcd9a2efd39d578d95f8299cc15a4d'

In [9]:
wandb.login(key=configs["wandb_key"])
wandb_logger = WandbLogger(project="live_project", config=configs)

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: avhrst (avhrst-org). Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\avhrs\.netrc


In [10]:
callbacks = [
    pl.pytorch.callbacks.ModelCheckpoint(
        dirpath="models",
        monitor="val/loss",
        save_top_k=1,
        mode="min"
    ),
    pl.pytorch.callbacks.EarlyStopping(
        monitor="val/loss",
        patience=20,
        mode="min"
    )
]

## Тренування моделі

In [11]:
trainer = pl.Trainer(max_epochs=configs["epochs"], accelerator="gpu", devices=[0],logger=wandb_logger, callbacks=callbacks)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [12]:
trainer.fit(model, train_loader, val_loader)

You are using a CUDA device ('NVIDIA GeForce RTX 4060 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


c:\Users\avhrs\Developer\python-learn\.venv\Lib\site-packages\lightning\pytorch\callbacks\model_checkpoint.py:653: Checkpoint directory C:\Users\avhrs\Developer\python-learn\models exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name       | Type                        | Params
-----------------------------------------------------------
0 | model      | Sequential                  | 145   
1 | train_mae  | MeanAbsoluteError           | 0     
2 | val_mae    | MeanAbsoluteError           | 0     
3 | train_msle | MeanSquaredLogError         | 0     
4 | val_msle   | MeanSquaredLogError         | 0     
5 | train_mape | MeanAbsolutePercentageError | 0     
6 | val_mape   | MeanAbsolutePercentageError | 0     
7 | train_msqe | MeanSquaredError            | 0     
8 | val_msqe   | MeanSquaredError            | 0     
9 | loss_fn    | MSELoss                     | 0     
-----------------------------------------------------------
145       Trainable params
0         

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\avhrs\Developer\python-learn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:441: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


c:\Users\avhrs\Developer\python-learn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:436: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.


In [ ]:
trainer.validate(model, val_loader)

In [ ]:
wandb.finish()